# Análisis Exploratorio de Datos

Dataset: Give Me Some Credit (Kaggle)
Target: `default` — default en los próximos 2 años

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '..')

from src.data.preprocess import cargar_datos, limpiar_datos, renombrar_columnas

df_raw = cargar_datos('data/raw/cs-training.csv')
df = renombrar_columnas(limpiar_datos(df_raw))
print(f"Registros antes de limpieza: {len(df_raw):,}")
print(f"Registros después de limpieza: {len(df):,}")
print(f"Tasa de default: {df['default'].mean():.2%}")

## Distribución del target

In [ ]:
counts = df['default'].value_counts()
print(counts)
print(f"\nDesbalance: {counts[0]/counts[1]:.1f}:1 (no-default:default)")
fig = px.pie(values=counts.values, names=['No Default', 'Default'],
             title='Distribución del target — Default en 2 años',
             color_discrete_sequence=['#2196F3', '#F44336'])
fig.show()

## NaN antes de la limpieza

In [ ]:
nan_info = df_raw.isna().sum()
nan_pct = (df_raw.isna().sum() / len(df_raw) * 100).round(2)
print(pd.DataFrame({'NaN': nan_info, '%': nan_pct})[nan_info > 0])

## Distribuciones de variables numéricas

In [ ]:
columnas_num = df.select_dtypes(include='number').columns.tolist()
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
axes = axes.flatten()
for i, col in enumerate(columnas_num[:11]):
    df[col].hist(ax=axes[i], bins=30, color='#1D5BA6', alpha=0.7)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('')
plt.suptitle('Distribuciones — Variables del Dataset', fontsize=14)
plt.tight_layout()
plt.show()

## Correlación con el target

In [ ]:
corr_target = df.corr(numeric_only=True)['default'].sort_values(ascending=False)
print("Correlaciones con default:")
print(corr_target.to_string())

fig = px.bar(
    x=corr_target.values[1:], y=corr_target.index[1:],
    orientation='h', title='Correlación de variables con Default',
    color=corr_target.values[1:], color_continuous_scale='RdBu_r',
    labels={'x': 'Correlación', 'y': 'Variable'}
)
fig.show()

## Distribución de ingreso por segmento de default

In [ ]:
fig = px.box(
    df[df['ingreso_mensual'] < df['ingreso_mensual'].quantile(0.99)],
    x='default', y='ingreso_mensual',
    title='Distribución de Ingreso Mensual por Default',
    labels={'default': 'Default', 'ingreso_mensual': 'Ingreso mensual'},
    color='default', color_discrete_map={0: '#2196F3', 1: '#F44336'}
)
fig.show()